In [2]:
import ee
from pathlib import Path
import shutil
import subprocess
import time

In [3]:
ee.Initialize(project='eecc-maureen')

#### Step 1. Data Extraction and Export (JRC Global River Flood Hazard Maps Version 2.1)

Source collection: `ee.ImageCollection("JRC/CEMS_GLOFAS/FloodHazard/v2_1")`

Bands: `RP10_depth`, `RP50_depth`, `RP100_depth`, `RP500_depth`

In [4]:
# Porto Alegre bounding box [minLon, minLat, maxLon, maxLat]
poa = ee.Geometry.Rectangle([-51.30344, -30.26945, -51.018852, -29.932474])

# JRC Global River Flood Hazard Maps v2.1
collection = ee.ImageCollection("JRC/CEMS_GLOFAS/FloodHazard/v2_1")
flood_hazard = ee.Image(collection.first())

bands = ["RP10_depth", "RP50_depth", "RP100_depth", "RP500_depth"]

tasks = []
for band in bands:
    img = (flood_hazard.select(band)
           .clip(poa)
           .reproject(crs="EPSG:3857", scale=90)
           .toFloat())

    export_name = f"poa_jrc_global_river_flood_hazard_maps_v2_1_{band}_90m"
    task = ee.batch.Export.image.toDrive(
        image=img,
        description=export_name,
        folder="gee_exports",
        fileNamePrefix=export_name,
        region=poa,
        scale=90,
        crs="EPSG:3857",
        maxPixels=1e13,
        fileFormat="GeoTIFF"
    )

    task.start()
    tasks.append(task)
    print(f"Started Drive export for {band}:", task.id)

Started Drive export for RP10_depth: XFBTBJBWISNXXBZZTB2GVWZJ
Started Drive export for RP50_depth: Q4RCHW3NLSZYMBWZKXGPL5GT
Started Drive export for RP100_depth: H7E3QGA72PLEXPZX3QRV6UHB
Started Drive export for RP500_depth: ZU6YSTJU5MWVGHDTJBUIRFAW


#### Step 2. Convert to COG and Generate Tiles

In [5]:
base = Path("data")
out_root = Path("out/jrc_global_river_flood_hazard_maps_v2_1")
bands = ["RP10_depth", "RP50_depth", "RP100_depth", "RP500_depth"]

In [9]:
# Preflight gdal2tiles runtime
gdal2tiles = shutil.which("gdal2tiles.py")
if not gdal2tiles:
    raise RuntimeError("gdal2tiles.py not found in PATH. Install GDAL first (e.g., `brew install gdal`).")

gdal2tiles_python = None
with open(gdal2tiles, "r", encoding="utf-8", errors="ignore") as f:
    first_line = f.readline().strip()
if first_line.startswith("#!"):
    parts = first_line[2:].split()
    if parts:
        if parts[0].endswith("env") and len(parts) > 1:
            gdal2tiles_python = shutil.which(parts[1])
        else:
            gdal2tiles_python = parts[0]
if not gdal2tiles_python:
    gdal2tiles_python = shutil.which("python3") or shutil.which("python")
if not gdal2tiles_python:
    raise RuntimeError("Could not determine Python interpreter for gdal2tiles.py")
subprocess.run([gdal2tiles_python, "-c", "import numpy"], check=True, capture_output=True)

for band in bands:
    band_slug = band.lower()
    in_tif = base / f"poa_jrc_global_river_flood_hazard_maps_v2_1_{band}_90m.tif"

    out_dir = out_root / band_slug
    cog_tif = out_dir / f"poa_jrc_global_river_flood_hazard_maps_v2_1_{band_slug}_90m_cog.tif"
    visual_tiles_dir = out_dir / "tiles_visual"
    value_tiles_dir = out_dir / "tiles_values"

    out_dir.mkdir(parents=True, exist_ok=True)

    # Convert to COG
    subprocess.run([
        "gdal_translate", str(in_tif), str(cog_tif),
        "-of", "COG",
        "-ot", "Float32",
        "-co", "COMPRESS=DEFLATE",
        "-co", "RESAMPLING=NEAREST",
        "-co", "OVERVIEWS=AUTO"
    ], check=True)
    print("Created COG:", cog_tif)

    # Visual tiles
    colorized_tif = out_dir / f"poa_jrc_global_river_flood_hazard_maps_v2_1_{band_slug}_colorized.tif"
    colors_txt = base / f"flood_hazard_{band_slug}_colors.txt"
    subprocess.run([
        "gdaldem", "color-relief", str(cog_tif), str(colors_txt), str(colorized_tif)
    ], check=True)

    visual_tiles_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
        str(colorized_tif), str(visual_tiles_dir)
    ], check=True)
    print(f"Visual tiles written to {band}:", visual_tiles_dir)

    # Value tiles: encode depth (meters) into RGB bytes for gdal2tiles compatibility.
    # Decode in app with: depth_m = (R + 256*G + 65536*B) / 100
    value_encoded_tif = out_dir / f"poa_jrc_global_river_flood_hazard_maps_v2_1_{band_slug}_value_encoded_rgb.tif"
    depth_scale = 100
    base_expr = f"rint(clip(A*{depth_scale},0,16777215)).astype(int64)"
    subprocess.run([
        "gdal_calc.py",
        "-A", str(cog_tif),
        "--calc", f"bitwise_and({base_expr},255)",
        "--calc", f"bitwise_and(right_shift({base_expr},8),255)",
        "--calc", f"bitwise_and(right_shift({base_expr},16),255)",
        "--type", "Byte",
        "--NoDataValue", "0",
        "--overwrite",
        "--outfile", str(value_encoded_tif)
    ], check=True)

    value_tiles_dir.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "gdal2tiles.py", "-r", "near", "-z", "8-15", "--xyz", "-w", "none",
        str(value_encoded_tif), str(value_tiles_dir)
    ], check=True)
    print(f"Value tiles written to {band}:", value_tiles_dir)

print("Done. Produced visual/value tiles for:", ", ".join(bands))
print("Decode value tiles with: depth_m = (R + 256*G + 65536*B) / 100")

Input file size is 353, 483
0...10...20...30...40...50...60...70...80...90...100 - done.
Created COG: out/jrc_global_river_flood_hazard_maps_v2_1/rp10_depth/poa_jrc_global_river_flood_hazard_maps_v2_1_rp10_depth_90m_cog.tif
0...10...20...30...40...50...60...70...80...90...100 - done.


Warning 1: Input dataset has no nodata value. Ignoring 'nv' entry in color palette
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0...10.

Generating Overview Tiles:


..20...30...40...50...60...70...80...90...100 - done.
Visual tiles written to RP10_depth: out/jrc_global_river_flood_hazard_maps_v2_1/rp10_depth/tiles_visual
0...10...20...30...40...50...60...70...80...90...100 - done.


...

.20...30...40...50...60...70...80...90...100 - done.
Visual tiles written to RP500_depth: out/jrc_global_river_flood_hazard_maps_v2_1/rp500_depth/tiles_visual
0...10...20...30...40...50...60...70...80...90...100 - done.


<string>:1: RuntimeWarning: invalid value encountered in cast
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0...10.

Generating Overview Tiles:


..20...30...40...50...60...70...80...90...100 - done.
Value tiles written to RP500_depth: out/jrc_global_river_flood_hazard_maps_v2_1/rp500_depth/tiles_values
Done. Produced visual/value tiles for: RP10_depth, RP50_depth, RP100_depth, RP500_depth
Decode value tiles with: depth_m = (R + 256*G + 65536*B) / 100
